# ELT — CPGF: Extract & Load

Ler os CSVs brutos de **2023, 2024 e 2025** e carregá-los em tabelas **Staging/Raw**, sem transformação prévia.

Usamos `csv` e `sqlite3`, da biblioteca padrão, para ler em lotes sem inferência de tipos e controlar a transação dos três anos.

Cada coluna mantém seu nome original e é armazenada como `TEXT`. Preservamos zeros à esquerda, máscaras, espaços, campos vazios (`''`), datas, vírgulas decimais, marcadores de sigilo e registros repetidos. Não há limpeza, renomeação, deduplicação ou conversão de valores. A transformação em dimensões e fatos fica para uma etapa posterior, dentro do banco.

## 1. Preparando o ambiente

Use Python 3.9 ou mais recente; não é necessário instalar pandas, SQLAlchemy ou dbt para esta etapa. Execute as células na ordem, de cima para baixo (no Colab: **Ambiente de execução > Executar tudo**; localmente: **Reiniciar kernel e executar tudo**). Rodar células fora de ordem deixa variáveis indefinidas nas células seguintes.

**No Colab**, a célula seguinte clona o repositório em `/content/projeto-integracao-dw` (ou atualiza o clone, se já existir) no ramo `RAMO_REPOSITORIO` e aponta `RAIZ_CONFIGURADA` para lá. Os scripts SQL da seção 6 precisam estar publicados nesse ramo no GitHub.

Os CSVs anuais completos não são versionados. Se `data/raw/{ano}_CPGF.csv` não existir, o notebook executa `scripts/extract_cpgf.py` automaticamente, que baixa os arquivos mensais do Portal da Transparência e os reúne em CSVs UTF-8 separados por `;` (leva alguns minutos). Para um teste rápido só com os arquivos versionados, use `MODO_AMOSTRA = True`; o resultado fica em outro banco.

In [ ]:
import subprocess
import sys
from pathlib import Path

EM_COLAB = 'google.colab' in sys.modules
URL_REPOSITORIO = 'https://github.com/fagpaf/projeto-integracao-dw'
RAMO_REPOSITORIO = 'sql-elt'  # troque para 'main' depois que o ramo for mesclado
PASTA_COLAB = Path('/content/projeto-integracao-dw')


def git(*argumentos, cwd=None):
    subprocess.run(['git', *argumentos], cwd=cwd, check=True)


if EM_COLAB:
    if (PASTA_COLAB / '.git').is_dir():
        # Clone de uma execução anterior: atualiza para o ramo configurado.
        git('fetch', 'origin', RAMO_REPOSITORIO, cwd=PASTA_COLAB)
        git('checkout', '-B', RAMO_REPOSITORIO, f'origin/{RAMO_REPOSITORIO}', cwd=PASTA_COLAB)
    else:
        git('clone', '--branch', RAMO_REPOSITORIO, URL_REPOSITORIO, str(PASTA_COLAB))
    if not any((PASTA_COLAB / 'scripts/sql/elt_transformations').glob('*.sql')):
        raise FileNotFoundError(
            f'O ramo {RAMO_REPOSITORIO!r} não tem os scripts em scripts/sql/elt_transformations/. '
            'Publique-os no GitHub ou ajuste RAMO_REPOSITORIO.'
        )
    print('Repositório pronto em', PASTA_COLAB, '- ramo', RAMO_REPOSITORIO)

In [ ]:
from pathlib import Path
from itertools import islice, zip_longest
import csv
import sqlite3
import time

RAIZ_CONFIGURADA = PASTA_COLAB if EM_COLAB else None
MODO_AMOSTRA = False
ANOS = (2023, 2024, 2025)
TAMANHO_LOTE = 10_000


def localizar_raiz(configurada=None):
    candidatos = [Path(configurada)] if configurada is not None else [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / 'scripts/extract_cpgf.py').is_file():
            return pasta.resolve()
    raise FileNotFoundError('Configure RAIZ_CONFIGURADA com a pasta do repositório.')


RAIZ = localizar_raiz(RAIZ_CONFIGURADA)
arquivos_brutos = {
    f'raw_cpgf_{ano}': RAIZ / 'data/raw' / (
        f'amostras/{ano}_amostra.csv' if MODO_AMOSTRA else f'{ano}_CPGF.csv'
    )
    for ano in ANOS
}
DESTINO = RAIZ / 'data/database' / (
    'cpgf_elt_amostra.sqlite' if MODO_AMOSTRA else 'cpgf_elt.sqlite'
)
print('Modo:', 'amostra' if MODO_AMOSTRA else 'completo')
print('Banco de destino:', DESTINO)

In [ ]:
faltando = [caminho.name for caminho in arquivos_brutos.values() if not caminho.is_file()]
if faltando and not MODO_AMOSTRA:
    print('CSVs ausentes:', ', '.join(faltando))
    print('Executando scripts/extract_cpgf.py (download do Portal da Transparência)...')
    anos = [str(min(ANOS)), str(max(ANOS))]
    resultado = subprocess.run([sys.executable, 'scripts/extract_cpgf.py', *anos], cwd=RAIZ)
    if resultado.returncode != 0:
        # Remove CSVs incompletos para a próxima execução não carregá-los como se estivessem completos.
        for caminho in arquivos_brutos.values():
            caminho.unlink(missing_ok=True)
        raise RuntimeError(
            'O download do Portal da Transparência falhou (veja a mensagem acima). '
            'Tente novamente mais tarde, coloque os arquivos {ano}_CPGF.csv em '
            f'{RAIZ / "data/raw"} ou use MODO_AMOSTRA = True.'
        )
else:
    print('CSVs de entrada já disponíveis.')

## 2. Definindo a leitura dos dados brutos

O SQLite não possui um schema `raw` separado neste arquivo: o prefixo `raw_` identifica a camada. Criamos uma tabela para cada arquivo anual, com as mesmas 15 colunas da fonte, sem colunas adicionais ou restrições de unicidade.

O leitor CSV retorna strings, inclusive para campos vazios e textos como `NA` e `NULL`. A leitura em lotes limita o uso de memória. Cabeçalhos inesperados, registros com quantidade incorreta de campos e erros de leitura interrompem a carga; nenhuma linha é ignorada silenciosamente. Essas conferências validam a estrutura, sem alterar o conteúdo.

In [ ]:
COLUNAS = [
    'CÓDIGO ÓRGÃO SUPERIOR', 'NOME ÓRGÃO SUPERIOR',
    'CÓDIGO ÓRGÃO', 'NOME ÓRGÃO',
    'CÓDIGO UNIDADE GESTORA', 'NOME UNIDADE GESTORA',
    'ANO EXTRATO', 'MÊS EXTRATO', 'CPF PORTADOR', 'NOME PORTADOR',
    'CNPJ OU CPF FAVORECIDO', 'NOME FAVORECIDO', 'TRANSAÇÃO',
    'DATA TRANSAÇÃO', 'VALOR TRANSAÇÃO',
]


def identificador(nome):
    # Aspas SQL permitem usar os nomes originais, inclusive espaços e acentos.
    return '"' + nome.replace('"', '""') + '"'


def ler_registros(caminho):
    with Path(caminho).open(encoding='utf-8-sig', newline='') as arquivo:
        leitor = csv.reader(arquivo, delimiter=';', strict=True)
        if next(leitor, None) != COLUNAS:
            raise ValueError(f'{caminho.name}: cabeçalho diferente do esperado.')
        for numero, registro in enumerate(leitor, start=1):
            if len(registro) != len(COLUNAS):
                raise ValueError(
                    f'{caminho.name}: registro {numero} tem {len(registro)} campos; '
                    f'esperados {len(COLUNAS)}.'
                )
            yield tuple(registro)


def ler_lotes(caminho, tamanho):
    if tamanho <= 0:
        raise ValueError('O tamanho do lote precisa ser positivo.')
    registros = ler_registros(caminho)
    try:
        while True:
            lote = list(islice(registros, tamanho))
            if not lote:
                break
            yield lote
    finally:
        registros.close()

## 3. Conferindo os arquivos de entrada

Todos os arquivos precisam estar disponíveis antes da carga. A conferência inicial lê o cabeçalho e o primeiro registro; as demais linhas serão verificadas durante a leitura completa. Arquivos sem registros também interrompem a execução.

In [ ]:
for tabela, caminho in arquivos_brutos.items():
    if not caminho.is_file():
        raise FileNotFoundError(
            f'Arquivo ausente: {caminho}. Execute scripts/extract_cpgf.py '
            'ou habilite MODO_AMOSTRA para usar as amostras.'
        )
    registros = ler_registros(caminho)
    try:
        if next(registros, None) is None:
            raise ValueError(f'{caminho.name}: arquivo sem registros.')
    finally:
        registros.close()
    print(f'{caminho.name} → {tabela} ({caminho.stat().st_size:,} bytes)')

## 4. Carregando a camada Raw

A carga é completa: as três tabelas Raw são recriadas em **uma única transação**. Reexecutar o notebook atualiza seu conteúdo, sem acumular cópias. Outras tabelas no banco permanecem intactas.

Antes do commit, comparamos a contagem e **cada campo de cada registro** armazenado com os CSVs, na ordem de inserção (`rowid` interno do SQLite). Isso confere também vazios, espaços e duplicatas. Se qualquer leitura, inserção ou conferência falhar, o rollback preserva as tabelas da carga anterior. Mantenha os CSVs inalterados durante a execução.

In [ ]:
def validar_tabela(conexao, tabela, caminho, quantidade):
    nome_sql = identificador(tabela)
    colunas_sql = ', '.join(map(identificador, COLUNAS))
    total_banco = conexao.execute(f'SELECT COUNT(*) FROM {nome_sql}').fetchone()[0]
    if total_banco != quantidade:
        raise ValueError(f'{tabela}: contagem divergente após a carga.')

    cursor = conexao.execute(f'SELECT {colunas_sql} FROM {nome_sql} ORDER BY rowid')
    origem = ler_registros(caminho)
    ausente = object()
    try:
        for numero, (esperado, armazenado) in enumerate(
            zip_longest(origem, cursor, fillvalue=ausente), start=1
        ):
            if esperado != armazenado:
                raise ValueError(f'{tabela}: conteúdo divergente no registro {numero}.')
    finally:
        origem.close()
        cursor.close()
    return total_banco


def carregar_raw(arquivos, destino, tamanho_lote=TAMANHO_LOTE):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    colunas_sql = ', '.join(map(identificador, COLUNAS))
    definicao_sql = ', '.join(f'{identificador(coluna)} TEXT' for coluna in COLUNAS)
    parametros = ', '.join('?' for _ in COLUNAS)
    resumo = []
    conexao = sqlite3.connect(destino)
    try:
        # BEGIN explícito inclui DROP/CREATE na mesma transação dos INSERTs.
        with conexao:
            conexao.execute('BEGIN')
            for tabela, caminho in arquivos.items():
                nome_sql = identificador(tabela)
                conexao.execute(f'DROP TABLE IF EXISTS {nome_sql}')
                conexao.execute(f'CREATE TABLE {nome_sql} ({definicao_sql})')
                quantidade = 0
                for lote in ler_lotes(caminho, tamanho_lote):
                    conexao.executemany(
                        f'INSERT INTO {nome_sql} ({colunas_sql}) VALUES ({parametros})', lote
                    )
                    quantidade += len(lote)
                if quantidade == 0:
                    raise ValueError(f'{caminho.name}: arquivo sem registros.')
                total_banco = validar_tabela(conexao, tabela, caminho, quantidade)
                resumo.append({
                    'tabela': tabela, 'arquivo': caminho.name,
                    'linhas_csv': quantidade, 'linhas_banco': total_banco,
                })
    finally:
        conexao.close()
    return resumo

In [ ]:
inicio = time.perf_counter()
resumo_carga = carregar_raw(arquivos_brutos, DESTINO)
print('Carga Raw concluída e validada:', DESTINO)
print('Tempo de carga e validação (s):', round(time.perf_counter() - inicio, 2))

## 5. Conferindo o resultado

As contagens abaixo vêm da execução atual e da consulta ao banco, sem usar totais fixos. A prévia consulta diretamente os campos originais, ainda como texto. O banco completo fica em `data/database/cpgf_elt.sqlite`; o modo amostra usa `cpgf_elt_amostra.sqlite`.

In [ ]:
print(f'{"Tabela":<20} {"Linhas CSV":>12} {"Linhas banco":>14}')
for item in resumo_carga:
    print(f'{item["tabela"]:<20} {item["linhas_csv"]:>12,} {item["linhas_banco"]:>14,}')
print('Total carregado:', sum(item['linhas_banco'] for item in resumo_carga))

conexao = sqlite3.connect(f'{DESTINO.resolve().as_uri()}?mode=ro', uri=True)
try:
    conexao.row_factory = sqlite3.Row
    for tabela in arquivos_brutos:
        print(f'\nPrévia de {tabela}:')
        consulta = (
            'SELECT "ANO EXTRATO", "MÊS EXTRATO", "DATA TRANSAÇÃO", "VALOR TRANSAÇÃO" '
            f'FROM {identificador(tabela)} ORDER BY rowid LIMIT 3'
        )
        for registro in conexao.execute(consulta):
            print(dict(registro))
finally:
    conexao.close()

## 6. Criando o esquema estrela dentro do banco

As dimensões e a fato (`DIM_*`, `FATO_TRANSACAO_CPGF`) são definidas em `scripts/sql/ddl/schema.sql` e vivem no mesmo arquivo `cpgf_elt.sqlite` da camada Raw — é isso que caracteriza o ELT: carregar primeiro e transformar depois, dentro do próprio banco, em vez de transformar em Python antes de carregar (como faz o `ETL.ipynb`).

Essas tabelas são recriadas do zero a cada execução, para que o notebook possa ser reexecutado sem acumular dados.

In [ ]:
DDL = RAIZ / 'scripts/sql/ddl/schema.sql'
PASTA_TRANSFORMACOES = RAIZ / 'scripts/sql/elt_transformations'
TABELAS_DIM = ['DIM_TEMPO', 'DIM_ORGAO', 'DIM_PORTADOR', 'DIM_FAVORECIDO', 'DIM_TIPO_TRANSACAO']

scripts_transformacao = sorted(PASTA_TRANSFORMACOES.glob('*.sql'))
if not scripts_transformacao:
    raise FileNotFoundError(f'Nenhum script encontrado em {PASTA_TRANSFORMACOES}.')
print('Scripts de transformação, na ordem de execução:')
for script in scripts_transformacao:
    print(' -', script.name)

## 7. Transformando e carregando as dimensões e a fato

Toda a limpeza de strings, conversão de números e datas, tratamento de sigilo/valores ausentes e o cálculo das chaves substitutas ficam dentro dos scripts SQL de `scripts/sql/elt_transformations/`. Eles leem apenas das tabelas `raw_cpgf_*` já carregadas e escrevem nas tabelas do esquema estrela — nenhuma transformação acontece em Python ou pandas, só orquestração e validação.

A carga é completa e roda em uma única transação: `FATO_TRANSACAO_CPGF` e as 5 `DIM_*` são recriadas antes de rodar os scripts, e os índices da fato são recriados só depois da carga — pelo mesmo motivo já explicado no `ETL.ipynb`: criá-los antes deixaria a inserção lenta.

In [ ]:
def recriar_esquema_estrela(conexao, ddl):
    # DROP na ordem que respeita as FKs: a fato antes das dimensões.
    conexao.execute('DROP TABLE IF EXISTS FATO_TRANSACAO_CPGF')
    for tabela in TABELAS_DIM:
        conexao.execute(f'DROP TABLE IF EXISTS {tabela}')
    conexao.executescript(Path(ddl).read_text(encoding='utf-8'))


def validar_esquema_estrela(conexao):
    contagens = {}
    for tabela in TABELAS_DIM + ['FATO_TRANSACAO_CPGF']:
        contagens[tabela] = conexao.execute(f'SELECT COUNT(*) FROM "{tabela}"').fetchone()[0]
    if any(quantidade == 0 for quantidade in contagens.values()):
        raise ValueError(f'Alguma tabela do esquema estrela ficou vazia: {contagens}')
    if conexao.execute('PRAGMA foreign_key_check').fetchall():
        raise ValueError('Há chaves estrangeiras inválidas.')
    if conexao.execute('PRAGMA integrity_check').fetchall() != [('ok',)]:
        raise ValueError('Falha na integridade SQLite.')

    # Confere contagem e soma em centavos: staging (origem) contra a fato carregada.
    total_staging = conexao.execute(
        'SELECT COUNT(*), SUM(valor_centavos) FROM stg_cpgf_transacoes'
    ).fetchone()
    total_fato = conexao.execute(
        'SELECT COUNT(*), SUM(CAST(ROUND(valor_transacao * 100) AS INTEGER)) FROM FATO_TRANSACAO_CPGF'
    ).fetchone()
    if total_staging != total_fato:
        raise ValueError(f'Totais divergentes entre staging e fato: {total_staging} != {total_fato}')

    # Mesma conferência, granular por mês do extrato e órgão (como no ETL.ipynb).
    divergencias = conexao.execute("""
        SELECT id_mes_extrato, id_orgao, COUNT(*), SUM(valor_centavos) FROM (
            SELECT st.id_mes_extrato AS id_mes_extrato, o.id_orgao AS id_orgao, st.valor_centavos AS valor_centavos
            FROM stg_cpgf_transacoes AS st
            JOIN DIM_ORGAO AS o
              ON o.codigo_unidade_gestora = st.codigo_unidade_gestora
             AND o.codigo_orgao = st.codigo_orgao
             AND o.codigo_orgao_superior = st.codigo_orgao_superior
        )
        GROUP BY id_mes_extrato, id_orgao
        EXCEPT
        SELECT id_mes_extrato, id_orgao, COUNT(*), CAST(SUM(ROUND(valor_transacao * 100)) AS INTEGER)
        FROM FATO_TRANSACAO_CPGF
        GROUP BY id_mes_extrato, id_orgao
    """).fetchall()
    if divergencias:
        raise ValueError(f'Totais por mês do extrato/órgão divergentes: {divergencias[:5]}')
    return contagens, total_staging, total_fato


def carregar_esquema_estrela(destino, ddl, scripts):
    conexao = sqlite3.connect(destino)
    try:
        conexao.execute('PRAGMA foreign_keys = ON')
        with conexao:
            conexao.execute('BEGIN')
            recriar_esquema_estrela(conexao, ddl)
            # Índices são dropados e recriados só depois da carga da fato.
            indices = conexao.execute(
                "SELECT name, sql FROM sqlite_master WHERE type='index' AND sql IS NOT NULL"
            ).fetchall()
            for nome, _ in indices:
                conexao.execute(f'DROP INDEX "{nome}"')
            for script in scripts:
                conexao.executescript(Path(script).read_text(encoding='utf-8'))
            for _, sql in indices:
                conexao.execute(sql)
            resultado = validar_esquema_estrela(conexao)
    finally:
        conexao.close()
    return resultado

In [ ]:
inicio_transformacao = time.perf_counter()
contagens, total_staging, total_fato = carregar_esquema_estrela(DESTINO, DDL, scripts_transformacao)
print('Esquema estrela carregado e validado em', DESTINO)
print('Tempo de transformação e validação (s):', round(time.perf_counter() - inicio_transformacao, 2))

## 8. Conferindo o resultado

As validações de contagem, integridade referencial e totais financeiros já rodaram dentro de `carregar_esquema_estrela`; se alguma divergisse, a carga teria sido interrompida e revertida. Aqui só exibimos os números conferidos.

In [ ]:
print(f'{"Tabela":<22} {"Registros":>12}')
for tabela, quantidade in contagens.items():
    print(f'{tabela:<22} {quantidade:>12,}')

print()
print(f'{"Etapa":<18} {"Transações":>12} {"Total centavos":>16}')
print(f'{"Staging (SQL)":<18} {total_staging[0]:>12,} {total_staging[1]:>16,}')
print(f'{"Fato (carregada)":<18} {total_fato[0]:>12,} {total_fato[1]:>16,}')